In [1]:
# Create a SQLite database and table

import sqlite3

# Connect to SQLite (or create if doesn't exist)
conn = sqlite3.connect('library.db')
cursor = conn.cursor()

# Create books table
cursor.execute('''
CREATE TABLE IF NOT EXISTS books (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    year INTEGER,
    available BOOLEAN DEFAULT 1
)
''')

conn.commit()
conn.close()

In [2]:
# Insert at least five book records

books = [
    ("1984", "George Orwell", 1949, True),
    ("To Kill a Mockingbird", "Harper Lee", 1960, True),
    ("The Great Gatsby", "F. Scott Fitzgerald", 1925, True),
    ("Brave New World", "Aldous Huxley", 1932, True),
    ("The Catcher in the Rye", "J.D. Salinger", 1951, True)
]

conn = sqlite3.connect('library.db')
cursor = conn.cursor()

cursor.executemany('''
INSERT INTO books (title, author, year, available) VALUES (?, ?, ?, ?)
''', books)

conn.commit()
conn.close()

In [3]:
# Query books by author

author_name = "George Orwell"

conn = sqlite3.connect('library.db')
cursor = conn.cursor()

cursor.execute('SELECT * FROM books WHERE author = ?', (author_name,))
results = cursor.fetchall()

for book in results:
    print(book)

conn.close()

(1, '1984', 'George Orwell', 1949, 1)


In [4]:
# Update book availability

book_id = 1  # let's say "1984" was borrowed

conn = sqlite3.connect('library.db')
cursor = conn.cursor()

cursor.execute('UPDATE books SET available = 0 WHERE id = ?', (book_id,))
conn.commit()
conn.close()

In [5]:
# Delete a record

book_id = 5  # let's remove "The Catcher in the Rye"

conn = sqlite3.connect('library.db')
cursor = conn.cursor()

cursor.execute('DELETE FROM books WHERE id = ?', (book_id,))
conn.commit()
conn.close()

In [6]:
# Recreate the same schema using SQLAlchemy ORM

from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.orm import declarative_base, sessionmaker

Base = declarative_base()
engine = create_engine('sqlite:///library_orm.db', echo=False)
Session = sessionmaker(bind=engine)
session = Session()

class Book(Base):
    __tablename__ = 'books'
    id = Column(Integer, primary_key=True)
    title = Column(String, nullable=False)
    author = Column(String, nullable=False)
    year = Column(Integer)
    available = Column(Boolean, default=True)

Base.metadata.create_all(engine)

In [8]:
# Insert and query records using ORM

# Insert books
books_orm = [
    Book(title="1984", author="George Orwell", year=1949),
    Book(title="To Kill a Mockingbird", author="Harper Lee", year=1960),
    Book(title="The Great Gatsby", author="F. Scott Fitzgerald", year=1925),
    Book(title="Brave New World", author="Aldous Huxley", year=1932),
    Book(title="The Catcher in the Rye", author="J.D. Salinger", year=1951)
]

session.add_all(books_orm)
session.commit()


In [9]:
# Query by author
for book in session.query(Book).filter_by(author="George Orwell").all():
    print(book.id, book.title, book.available)



1 1984 False
5 1984 True


In [10]:
# Update availability
book_to_update = session.query(Book).filter_by(id=1).first()
book_to_update.available = False
session.commit()


In [11]:
# Delete a record
book_to_delete = session.query(Book).filter_by(id=5).first()
session.delete(book_to_delete)
session.commit()